# One SPSA training iteration with PyTorch

Use the existing game evaluator as a PyTorch `nn.Linear` layer. Play two nearby weight configurations, estimate a gradient from their results, and let `torch.optim.SGD` update the weights.

Start JupyterLab from `part_b` using the command in `training/README.md`. This notebook runs from `training/spsa`. Use **Python (Cascade)** and run cells in order with **Shift+Enter**.

SPSA is not built into PyTorch. The probe and gradient estimate are the algorithm-specific part. PyTorch manages parameters, updates, learning-rate scheduling and checkpoints. Game results come from external processes, so there is no autograd path from a match score to the weights and no `loss.backward()` in this algorithm.

This notebook tunes weights from **newly played games**. It does not train on the saved feature archive; that archive is loaded only for the example forward pass. Agents choose all moves, including their initial placements.


In [ ]:
from dataclasses import asdict
from pathlib import Path

import torch
from torch import nn

from Agents.common.features.movement import FEATURE_NAMES
from GameGeneration.generate import save_json
from training.DataExtraction.extract_features import FeatureGame, load_features
from training.spsa.config import SPSAConfig, load_config
from training.spsa.matches import play_batch
from training.spsa.model import make_model, model_weights, to_weights
from training.spsa.optimizer import estimate_gradient, propose
from training.spsa.trainer import save_checkpoint

config: SPSAConfig = load_config(Path("config.toml"))
iteration: int = 0

## Model parameters

`make_model` initialises a standard `nn.Linear(8, 1, bias=False)` from our configured weights. `model.weight` is an `nn.Parameter` tensor with shape $(1,8)$ and dtype `torch.float64`, matching the previous trainer's coefficient precision.

`model_weights` exports the tensor to named Python floats for the existing playing agent. The playing agent continues using its current feature calculations and search.

Before training, inspect a forward pass on one recorded Red position. The stored features are float32; convert them to the model's float64 dtype before calling the layer. This example position is not a training target.


In [ ]:
model: nn.Linear = make_model(config.weights)
print(model)
print(model.weight.shape, model.weight.dtype)
print(model_weights(model))

feature_path: Path = Path("../../logs/shortBudget-TeacherL3L4/features/game-000001.npz")
game: FeatureGame = load_features(feature_path)
sample_features: torch.Tensor = torch.from_numpy(game["features"][8, 0].copy()).to(
    dtype=model.weight.dtype,
).unsqueeze(0)
with torch.no_grad():
    sample_score: torch.Tensor = model(sample_features)
print("Input:", sample_features.shape, sample_features.dtype)
print("Score:", sample_score)


## Built-in optimiser and scheduler

Use [`SGD`](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html) with `maximize=True` because a higher match score is better. Momentum and weight decay stay at their defaults of zero, so frozen coordinates with zero gradients remain fixed.

[`LambdaLR`](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.LambdaLR.html) handles the configured learning-rate decay. Call the scheduler after the optimiser update.

In [ ]:
optimizer: torch.optim.SGD = torch.optim.SGD(
    model.parameters(), lr=config.learning_rate, maximize=True,
)
scheduler: torch.optim.lr_scheduler.LambdaLR = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: (step + 1) ** (-config.learning_decay),
)
print("Learning rate:", optimizer.param_groups[0]["lr"])

## Make two nearby weight configurations

Each tunable coordinate gets a random sign. Material and alignment are frozen by the default configuration and have direction zero.

$$
\mathbf w^+=\mathbf w+c_k\boldsymbol\Delta,
\qquad
\mathbf w^-=\mathbf w-c_k\boldsymbol\Delta.
$$

The helper moves the center inward near bounds to keep both probes symmetric and feasible. The vectors below have shape $(8,)$ and dtype `torch.float64`.

In [ ]:
proposal = propose(model, config, iteration)
weights_before: dict[str, float] = model_weights(model)
print("Plus:", to_weights(proposal.plus))
print("Minus:", to_weights(proposal.minus))
print("Directions:", list(zip(FEATURE_NAMES, proposal.direction.tolist())))

## Play the actual games

This cell runs the configured games; it is not a demonstration on fabricated scores. The default uses eight games: both probes, with two colour-swapped game pairs each against the fixed-weight control. It can take several minutes.

Choose a fresh output name when starting another notebook run. Each probe uses the same opponent and seed schedule. The runner saves referee logs, results, raw board archives and extracted features.

The objective is mean match score:

$$
J(\mathbf w)=\frac{W+\tfrac12D}{N}.
$$

In [ ]:
output: Path = Path("../../logs/spsa-notebook-001")
output.mkdir(parents=True)
save_json(output / "config.json", asdict(config))
save_checkpoint(output / "checkpoint.pt", model, optimizer, scheduler, config, 0)

iteration_output: Path = output / "iteration-0001"
plus_result = play_batch(to_weights(proposal.plus), config, iteration_output / "plus", iteration)
minus_result = play_batch(to_weights(proposal.minus), config, iteration_output / "minus", iteration)
print("Plus:", plus_result)
print("Minus:", minus_result)

## Estimate the gradient

With signs chosen from $\{-1,+1\}$, the SPSA gradient estimate is

$$
\widehat{\mathbf g}
=\frac{J(\mathbf w^+)-J(\mathbf w^-)}{2c_k}\boldsymbol\Delta.
$$

`estimate_gradient` implements this expression in one tensor operation. The gradient is estimated from game results. It is not a derivative computed by autograd. Equal scores give a zero estimate for this iteration.

In [ ]:
gradient: torch.Tensor = estimate_gradient(proposal, plus_result["score"], minus_result["score"])
print(gradient.shape, gradient.dtype)
list(zip(FEATURE_NAMES, gradient.tolist()))

## Give the estimate to SGD

`optimizer.zero_grad` clears any old gradients. Assign our estimate to `model.weight.grad`, then call `optimizer.step()` to update the parameters. PyTorch's `no_grad` context is used for explicit center placement and bound projection.

The default material and alignment coordinates stay fixed. This is the same update used by the command-line trainer, shown explicitly here for inspection.

In [ ]:
optimizer.zero_grad(set_to_none=True)
with torch.no_grad():
    model.weight.copy_(proposal.center.unsqueeze(0))
model.weight.grad = gradient.unsqueeze(0)
optimizer.step()

active: torch.Tensor = proposal.direction != 0
with torch.no_grad():
    model.weight[0, active] = model.weight[0, active].clamp(config.lower_bound, config.upper_bound)
scheduler.step()

weights_after: dict[str, float] = model_weights(model)
[(name, weights_before[name], weights_after[name]) for name in FEATURE_NAMES]

## Save PyTorch state

PyTorch's `state_dict` records the model, optimiser and scheduler state. Saving all three makes the run resumable. The helper uses `torch.save` and also writes `weights.json` for the game agent.

[`Saving and loading models`](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html) describes this standard PyTorch workflow. These new `.pt` checkpoints are separate from the earlier NumPy trainer's JSON checkpoints.

In [ ]:
save_json(iteration_output / "update.json", {
    "iteration": 1,
    "weights_before": weights_before,
    "estimated_gradient": gradient.tolist(),
    "results": {"plus": plus_result, "minus": minus_result},
    "weights_after": weights_after,
})
save_checkpoint(output / "checkpoint.pt", model, optimizer, scheduler, config, 1)

checkpoint = torch.load(output / "checkpoint.pt", map_location="cpu", weights_only=True)
print(checkpoint.keys())
print("Completed iterations:", checkpoint["completed_iterations"])
checkpoint["model_state_dict"]

Inspect the two match scores, the estimated gradient and the before/after weights. If the scores tied, the gradient is zero; this is a valid result, not evidence of a broken optimiser.

This one iteration cannot establish that the resulting weights play better. Continue training and evaluate fixed weights in separate matches before drawing that conclusion.